# Day 11 - UDFs and Built-in Functions

**Topic:** UDFs and Built-in Functions  
**Main environment:** Microsoft Fabric Notebook + Fabric Lakehouse  
**Focus:** เข้าใจว่าเมื่อไรควรใช้ PySpark built-in functions และเมื่อไรจึงควรใช้ Python UDF อย่างระมัดระวัง

วันนี้เราจะเรียนเรื่องการ transform ข้อมูลด้วย PySpark built-in functions ก่อน แล้วค่อยดูว่า Python UDF คืออะไร ใช้อย่างไร และทำไมในงาน Spark/Lakehouse pipeline จริงควรใช้ UDF เป็นทางเลือกหลัง ๆ ไม่ใช่ทางเลือกแรก

## Goal of the day

1. อธิบายได้ว่า **built-in functions** และ **Python UDF** ต่างกันอย่างไร
2. ใช้ built-in functions เช่น `trim`, `lower`, `regexp_replace`, `when`, `concat_ws` เพื่อ clean / derive columns ได้
3. เขียน Python UDF แบบง่าย ๆ พร้อมกำหนด return type ได้
4. อ่าน `.explain()` เพื่อเริ่มสังเกตว่า UDF ทำให้ plan ต่างจาก built-in functions อย่างไร
5. เข้าใจ production mindset ว่า **ควรใช้ built-in functions ก่อน UDF** เมื่อทำได้

## Why it matters in production

ใน production Spark pipeline เราอาจเจอ business logic ที่ต้อง transform ข้อมูลหลายรูปแบบ เช่น standardize status, clean payment method, derive amount band, mask email หรือ map rule เฉพาะทาง

เรื่องนี้สำคัญเพราะ:

- built-in functions เป็น expression ที่ Spark เข้าใจและ optimize ได้ดีกว่า
- Python UDF มักทำให้ Spark ต้องส่งข้อมูลไปประมวลผลใน Python process เพิ่ม serialization overhead
- UDF อาจทำให้ query plan อ่านยากขึ้น และ Spark optimize logic ข้างใน function ได้น้อยลง
- ถ้าใช้ UDF กับข้อมูลใหญ่โดยไม่จำเป็น pipeline อาจช้าลงทั้งที่ logic ดูง่ายมาก
- การเลือก function ให้ถูกตั้งแต่แรกช่วยให้ code maintain ง่ายและ debug ง่ายขึ้น

Production takeaway:

> ใช้ built-in functions ก่อนเสมอ ถ้า logic ยังเขียนด้วย PySpark Column expression ได้ อย่าเพิ่งรีบใช้ UDF

## Key concepts

| Concept | Meaning |
|---|---|
| Built-in functions | Function สำเร็จรูปใน `pyspark.sql.functions` เช่น `lower`, `trim`, `when`, `regexp_replace` ที่ Spark เข้าใจและ optimize ได้ |
| Column expression | Logic ที่ให้ผลลัพธ์เป็น Column object ซึ่งสร้างจาก column, operator และ built-in functions เช่น `F.col("amount") > 0`, `F.when(...)` |
| Python UDF | User Defined Function ที่เขียนด้วย Python แล้ว register ให้ Spark เรียกใช้กับ column ได้ |
| Serialization overhead | Cost จากการแปลง/ส่งข้อมูลระหว่าง Spark JVM process กับ Python process |
| Catalyst optimizer | Optimizer ของ Spark ที่ช่วยปรับ logical plan / physical plan สำหรับ built-in expressions ได้ดี |
| Return type | Data type ที่ต้องระบุให้ UDF เช่น `T.StringType()` หรือ `T.IntegerType()` |
| Null handling | การจัดการค่า null ใน function เพื่อไม่ให้ UDF error หรือให้ผลลัพธ์ผิด |

## Concept explanation

### Built-in functions คืออะไร?

Built-in functions คือ function ที่ Spark เตรียมไว้ให้ใช้กับ DataFrame columns เช่น:

```python
F.trim(F.col("customer_name"))
F.lower(F.col("status"))
F.when(F.col("amount") >= 1000, "high").otherwise("normal")
```

ข้อดีคือ Spark มองเห็น logic เหล่านี้ใน execution plan และสามารถ optimize ได้ดีกว่า Python function ทั่วไป

### Python UDF คืออะไร?

Python UDF คือ function ที่เราเขียนเองด้วย Python แล้วให้ Spark apply กับแต่ละ row/column เช่น:

```python
def classify_amount(amount):
    if amount is None:
        return "unknown"
    if amount >= 3000:
        return "high"
    return "normal"

classify_amount_udf = F.udf(classify_amount, T.StringType())
```

UDF มีประโยชน์เมื่อ logic ซับซ้อนมาก หรือไม่มี built-in function รองรับจริง ๆ แต่ต้องระวัง performance และ maintainability

### หลักคิดในการเลือกใช้

ลำดับที่ควรคิดก่อนเขียน UDF:

1. มี built-in function ใช้ได้ไหม?
2. เขียนเป็น `when` / `otherwise` ได้ไหม?
3. เขียนเป็น `F.expr()` หรือ Spark SQL expression ได้ไหม?
4. ถ้ายังไม่ได้จริง ๆ ค่อยใช้ UDF และต้องกำหนด return type + handle null ให้ดี

> UDF ไม่ได้ผิด แต่ถ้าใช้ UDF กับ logic ง่าย ๆ ที่ built-in functions ทำได้อยู่แล้ว ควรพิจารณาเปลี่ยนกลับมาใช้ built-in functions ก่อน


## Mermaid diagram: Built-in Functions vs Python UDF

```mermaid
flowchart LR
    A[Input DataFrame] --> B{Transformation Logic}
    B --> C[Built-in functions / Column expressions]
    C --> D[Catalyst can optimize plan]
    D --> E[Spark engine executes efficiently]

    B --> F[Python UDF]
    F --> G[Serialize data to Python worker]
    G --> H[Run custom Python function]
    H --> I[Return result back to Spark]
    I --> J[Less visible to optimizer]
```

Key idea:

> Built-in functions ให้ Spark เห็น logic ชัดกว่า ส่วน Python UDF เหมือนกล่องดำที่ Spark optimize ข้างในได้น้อยกว่า

## Setup / imports

In [2]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 4, Finished, Available, Finished, False)

## Create mock data

ชุดข้อมูลวันนี้จำลอง transaction data ที่มี text ไม่สะอาด, payment method หลาย format, amount หลายช่วง และ email ที่ต้อง derive domain / masked value

In [3]:
transactions_data = [
    ("T001", 101, "  Alice Wong  ", "Credit Card", "PRD-INS-001", 1200.50, " SUCCESS ", "SAVE10", "alice.wong@example.com", "web"),
    ("T002", 102, "bob smith", "prompt-pay", "PRD-LOAN-002", 250.00, "success", None, "bob.smith@example.com", "mobile"),
    ("T003", 103, "CHARLIE TAN", "Bank Transfer", "PRD-INS-003", 3400.00, "Success", "VIP30", "charlie.tan@company.co.th", "branch"),
    ("T004", 104, "Dana Lee", "cash", "SRV-CLAIM-001", 0.00, "FAILED", "", "dana.lee@example.com", "branch"),
    ("T005", 105, None, "e wallet", "PRD-CARD-004", 780.00, "Pending", "WELCOME", None, "mobile"),
    ("T006", 106, "Eve Kim", "CREDIT_CARD", "PRD-INV-005", 5600.00, "success", "VIP50", "eve.kim@example.com", "web"),
    ("T007", 107, "Frank Nu", "promptpay", "PRD-INS-006", -100.00, "failed", None, "frank.nu@example.com", "mobile"),
    ("T008", 108, "Grace Lim", "bank-transfer", "SRV-SUPPORT-002", 980.00, "SUCCESS", "SAVE10", "grace.lim@company.co.th", "web"),
]

transaction_schema = T.StructType([
    T.StructField("transaction_id", T.StringType(), False),
    T.StructField("customer_id", T.IntegerType(), False),
    T.StructField("customer_name_raw", T.StringType(), True),
    T.StructField("payment_method_raw", T.StringType(), True),
    T.StructField("product_code", T.StringType(), True),
    T.StructField("amount", T.DoubleType(), True),
    T.StructField("status_raw", T.StringType(), True),
    T.StructField("coupon_code", T.StringType(), True),
    T.StructField("email", T.StringType(), True),
    T.StructField("source_system", T.StringType(), True),
])

df_transactions = spark.createDataFrame(transactions_data, transaction_schema)

df_transactions.show(truncate=False)
df_transactions.printSchema()

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 5, Finished, Available, Finished, False)

+--------------+-----------+-----------------+------------------+---------------+------+----------+-----------+-------------------------+-------------+
|transaction_id|customer_id|customer_name_raw|payment_method_raw|product_code   |amount|status_raw|coupon_code|email                    |source_system|
+--------------+-----------+-----------------+------------------+---------------+------+----------+-----------+-------------------------+-------------+
|T001          |101        |  Alice Wong     |Credit Card       |PRD-INS-001    |1200.5| SUCCESS  |SAVE10     |alice.wong@example.com   |web          |
|T002          |102        |bob smith        |prompt-pay        |PRD-LOAN-002   |250.0 |success   |NULL       |bob.smith@example.com    |mobile       |
|T003          |103        |CHARLIE TAN      |Bank Transfer     |PRD-INS-003    |3400.0|Success   |VIP30      |charlie.tan@company.co.th|branch       |
|T004          |104        |Dana Lee         |cash              |SRV-CLAIM-001  |0.0   |

## PySpark code examples

ใน section นี้จะเริ่มจาก built-in functions ก่อน แล้วค่อยเขียน UDF เพื่อเปรียบเทียบทั้ง readability และ execution plan mindset

### Example 1: Clean and derive columns with built-in functions

ตัวอย่างนี้ใช้ built-in functions เพื่อ standardize columns ที่เจอบ่อยในงาน Data Engineering:

- trim / normalize customer name
- normalize status และ payment method
- extract product group
- derive email domain
- create `amount_band`
- create masked email แบบง่าย ๆ

In [4]:
df_builtin_cleaned = (
    df_transactions
    .withColumn("customer_name", F.initcap(F.trim(F.col("customer_name_raw"))))
    .withColumn("status", F.lower(F.trim(F.col("status_raw"))))
    .withColumn(
        "payment_method",
        F.regexp_replace(F.lower(F.trim(F.col("payment_method_raw"))), r"[\s-]+", "_")  # replace one or more spaces/hyphens with underscore
    )
    .withColumn("product_group", F.regexp_extract(F.col("product_code"), r"^([A-Z]+)", 1))  # extract leading uppercase letters from product_code
    .withColumn("email_domain", F.lower(F.regexp_extract(F.col("email"), r"@(.+)$", 1)))  # extract everything after @ as email domain
    .withColumn(
        "coupon_code_clean",
        F.when(F.trim(F.col("coupon_code")) == "", F.lit(None)).otherwise(F.upper(F.trim(F.col("coupon_code"))))
    )
    .withColumn(
        "amount_band",
        F.when(F.col("amount").isNull(), F.lit("unknown"))
         .when(F.col("amount") < 0, F.lit("invalid"))
         .when(F.col("amount") == 0, F.lit("zero"))
         .when(F.col("amount") >= 3000, F.lit("high"))
         .when(F.col("amount") >= 1000, F.lit("medium"))
         .otherwise(F.lit("low"))
    )
    .withColumn(
        "masked_email",
        F.when(
            F.col("email").isNotNull(),
            F.concat(F.lit("***@"), F.col("email_domain"))
        ).otherwise(F.lit(None))
    )
    .select(
        "transaction_id",
        "customer_id",
        "customer_name",
        "status",
        "payment_method",
        "product_group",
        "amount",
        "amount_band",
        "coupon_code_clean",
        "email_domain",
        "masked_email",
        "source_system"
    )
)

df_builtin_cleaned.show(truncate=False)

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 6, Finished, Available, Finished, False)

+--------------+-----------+-------------+-------+--------------+-------------+------+-----------+-----------------+-------------+-----------------+-------------+
|transaction_id|customer_id|customer_name|status |payment_method|product_group|amount|amount_band|coupon_code_clean|email_domain |masked_email     |source_system|
+--------------+-----------+-------------+-------+--------------+-------------+------+-----------+-----------------+-------------+-----------------+-------------+
|T001          |101        |Alice Wong   |success|credit_card   |PRD          |1200.5|medium     |SAVE10           |example.com  |***@example.com  |web          |
|T002          |102        |Bob Smith    |success|prompt_pay    |PRD          |250.0 |low        |NULL             |example.com  |***@example.com  |mobile       |
|T003          |103        |Charlie Tan  |success|bank_transfer |PRD          |3400.0|high       |VIP30            |company.co.th|***@company.co.th|branch       |
|T004          |104   

### Example 2: Use `when` / `otherwise` instead of UDF for rule mapping

หลาย rule ที่ดูเหมือน business logic สามารถเขียนด้วย built-in Column expression ได้ เช่น payment channel grouping

In [5]:
df_payment_grouped = (
    df_builtin_cleaned
    .withColumn(
        "payment_channel_group",
        F.when(F.col("payment_method").isin("credit_card"), F.lit("card"))
         .when(F.col("payment_method").isin("prompt_pay", "promptpay"), F.lit("instant_payment"))
         .when(F.col("payment_method").isin("bank_transfer"), F.lit("bank"))
         .when(F.col("payment_method").isin("e_wallet"), F.lit("wallet"))
         .otherwise(F.lit("other"))
    )
)

df_payment_grouped.select(
    "transaction_id",
    "payment_method",
    "payment_channel_group",
    "amount",
    "amount_band"
).show(truncate=False)

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 7, Finished, Available, Finished, False)

+--------------+--------------+---------------------+------+-----------+
|transaction_id|payment_method|payment_channel_group|amount|amount_band|
+--------------+--------------+---------------------+------+-----------+
|T001          |credit_card   |card                 |1200.5|medium     |
|T002          |prompt_pay    |instant_payment      |250.0 |low        |
|T003          |bank_transfer |bank                 |3400.0|high       |
|T004          |cash          |other                |0.0   |zero       |
|T005          |e_wallet      |wallet               |780.0 |low        |
|T006          |credit_card   |card                 |5600.0|high       |
|T007          |promptpay     |instant_payment      |-100.0|invalid    |
|T008          |bank_transfer |bank                 |980.0 |low        |
+--------------+--------------+---------------------+------+-----------+



### Example 3: Write the same logic as a Python UDF

ตัวอย่างนี้เขียน `amount_band` ด้วย Python UDF เพื่อให้เห็น syntax และข้อควรระวัง

สิ่งสำคัญ:

- ต้องกำหนด return type เช่น `T.StringType()`
- ต้อง handle null เองใน Python function
- เหมาะกับ logic ที่ built-in functions ทำไม่ได้จริง ๆ มากกว่าใช้เป็น default

In [6]:
def classify_amount_band(amount):
    if amount is None:
        return "unknown"
    if amount < 0:
        return "invalid"
    if amount == 0:
        return "zero"
    if amount >= 3000:
        return "high"
    if amount >= 1000:
        return "medium"
    return "low"

classify_amount_band_udf = F.udf(classify_amount_band, T.StringType())

df_udf_band = (
    df_transactions
    .withColumn("amount_band_udf", classify_amount_band_udf(F.col("amount")))
    .select("transaction_id", "amount", "amount_band_udf")
)

df_udf_band.show()

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 9, Finished, Available, Finished, False)

+--------------+------+---------------+
|transaction_id|amount|amount_band_udf|
+--------------+------+---------------+
|          T001|1200.5|         medium|
|          T002| 250.0|            low|
|          T003|3400.0|           high|
|          T004|   0.0|           zero|
|          T005| 780.0|            low|
|          T006|5600.0|           high|
|          T007|-100.0|        invalid|
|          T008| 980.0|            low|
+--------------+------+---------------+



### Example 4: Compare built-in result and UDF result

บางครั้ง UDF ให้ผลลัพธ์เหมือน built-in function แต่ไม่ได้แปลว่าควรใช้ UDF เสมอไป

ในตัวอย่างนี้เราจะ compare output เพื่อยืนยันว่า logic ให้ผลเหมือนกัน

In [7]:
df_compare_amount_band = (
    df_builtin_cleaned
    .select("transaction_id", "amount", "amount_band")  # keep only columns needed for comparison
    .join(df_udf_band, on=["transaction_id", "amount"], how="inner")
    .withColumn("is_same_result", F.col("amount_band") == F.col("amount_band_udf"))
    .orderBy("transaction_id")
)

df_compare_amount_band.show()

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 12, Finished, Available, Finished, False)

+--------------+------+-----------+---------------+--------------+
|transaction_id|amount|amount_band|amount_band_udf|is_same_result|
+--------------+------+-----------+---------------+--------------+
|          T001|1200.5|     medium|         medium|          true|
|          T002| 250.0|        low|            low|          true|
|          T003|3400.0|       high|           high|          true|
|          T004|   0.0|       zero|           zero|          true|
|          T005| 780.0|        low|            low|          true|
|          T006|5600.0|       high|           high|          true|
|          T007|-100.0|    invalid|        invalid|          true|
|          T008| 980.0|        low|            low|          true|
+--------------+------+-----------+---------------+--------------+



### Example 5: Inspect plans with `.explain()`

ให้ดูความต่างเชิง mindset:

- Built-in version: Spark มักเห็น logic เป็น native Spark expressions เช่น `CASE WHEN`, `lower`, `trim`, `regexp_replace`
- UDF version: Spark มักเห็น Python UDF step เช่น `BatchEvalPython` ทำให้ optimize logic ข้างในได้จำกัดกว่า built-in functions

ไม่ต้องอ่าน plan ได้ทั้งหมดในวันนี้ แค่เริ่มสังเกตว่า UDF ทำให้ plan มี boundary เพิ่มขึ้น

In [8]:
print("Built-in function plan:")
df_builtin_cleaned.select("transaction_id", "amount", "amount_band").explain(mode="formatted")  # inspect only the columns needed for amount band logic

print("Python UDF plan:")
df_udf_band.explain(mode="formatted")

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 25, Finished, Available, Finished, False)

Built-in function plan:
== Physical Plan ==
* Project (2)
+- * Scan ExistingRDD (1)


(1) Scan ExistingRDD [codegen id : 1]
Output [10]: [transaction_id#726, customer_id#727, customer_name_raw#728, payment_method_raw#729, product_code#730, amount#731, status_raw#732, coupon_code#733, email#734, source_system#735]
Arguments: [transaction_id#726, customer_id#727, customer_name_raw#728, payment_method_raw#729, product_code#730, amount#731, status_raw#732, coupon_code#733, email#734, source_system#735], MapPartitionsRDD[44] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Project [codegen id : 1]
Output [3]: [transaction_id#726, amount#731, CASE WHEN isnull(amount#731) THEN unknown WHEN (amount#731 < 0.0) THEN invalid WHEN (amount#731 = 0.0) THEN zero WHEN (amount#731 >= 3000.0) THEN high WHEN (amount#731 >= 1000.0) THEN medium ELSE low END AS amount_band#874]
Input [10]: [transaction_id#726, customer_id#727, customer_name_raw#728, paym

### Example 6: Use UDF only when custom logic is really needed

ตัวอย่างนี้สมมติว่ามี legacy priority rule ที่เป็น Python function เดิมอยู่แล้ว หรือ logic ซับซ้อนมากจนเขียนด้วย built-in expressions ไม่คุ้ม

ใน production จริง ถ้า rule ยังเขียนด้วย `when` / `otherwise` ได้ชัดเจนกว่า ก็ควรใช้ built-in approach ก่อน

In [9]:
def legacy_priority_rule(status, amount, source_system):
    if status is None or amount is None:
        return "review"
    status_normalized = status.strip().lower()
    if amount < 0:
        return "invalid"
    if status_normalized == "failed":
        return "review"
    if status_normalized == "success" and amount >= 3000 and source_system in ["web", "branch"]:
        return "p1"
    if status_normalized == "pending":
        return "p2"
    return "standard"

legacy_priority_udf = F.udf(legacy_priority_rule, T.StringType())

df_with_legacy_priority = (
    df_transactions
    .withColumn(
        "legacy_priority",
        legacy_priority_udf(
            F.col("status_raw"),
            F.col("amount"),
            F.col("source_system")
        )
    )
    .select("transaction_id", "status_raw", "amount", "source_system", "legacy_priority")
)

df_with_legacy_priority.show(truncate=False)

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 26, Finished, Available, Finished, False)

+--------------+----------+------+-------------+---------------+
|transaction_id|status_raw|amount|source_system|legacy_priority|
+--------------+----------+------+-------------+---------------+
|T001          | SUCCESS  |1200.5|web          |standard       |
|T002          |success   |250.0 |mobile       |standard       |
|T003          |Success   |3400.0|branch       |p1             |
|T004          |FAILED    |0.0   |branch       |review         |
|T005          |Pending   |780.0 |mobile       |p2             |
|T006          |success   |5600.0|web          |p1             |
|T007          |failed    |-100.0|mobile       |invalid        |
|T008          |SUCCESS   |980.0 |web          |standard       |
+--------------+----------+------+-------------+---------------+



### Example 7: Write cleaned function output to a Lakehouse table

Cell ด้านล่างเขียนผลลัพธ์จาก built-in functions เป็น table ชื่อ `day11_transaction_function_features`

> หมายเหตุ: ต้อง attach Lakehouse เข้ากับ Fabric Notebook ก่อนรัน cell นี้ และ user ต้องมี permission ใน Lakehouse

In [10]:
table_name = "day11_transaction_function_features"

(
    df_payment_grouped
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable(table_name)
)

print(f"Table written successfully: {table_name}")

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 27, Finished, Available, Finished, False)

Table written successfully: day11_transaction_function_features


### Example 8: Read table back from Lakehouse

หลังจาก write table แล้ว สามารถ read กลับด้วย `spark.read.table()` เพื่อ validate output

In [11]:
df_saved_features = spark.read.table("day11_transaction_function_features")

df_saved_features.select(
    "transaction_id",
    "customer_name",
    "status",
    "payment_method",
    "payment_channel_group",
    "amount_band"
).show(truncate=False)

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 28, Finished, Available, Finished, False)

+--------------+-------------+-------+--------------+---------------------+-----------+
|transaction_id|customer_name|status |payment_method|payment_channel_group|amount_band|
+--------------+-------------+-------+--------------+---------------------+-----------+
|T003          |Charlie Tan  |success|bank_transfer |bank                 |high       |
|T008          |Grace Lim    |success|bank_transfer |bank                 |low        |
|T001          |Alice Wong   |success|credit_card   |card                 |medium     |
|T007          |Frank Nu     |failed |promptpay     |instant_payment      |invalid    |
|T002          |Bob Smith    |success|prompt_pay    |instant_payment      |low        |
|T006          |Eve Kim      |success|credit_card   |card                 |high       |
|T004          |Dana Lee     |failed |cash          |other                |zero       |
|T005          |NULL         |pending|e_wallet      |wallet               |low        |
+--------------+-------------+--

## Quick recap

| Question | Answer |
|---|---|
| ควรใช้ built-in functions หรือ UDF ก่อน? | ใช้ built-in functions ก่อน ถ้า logic ทำได้ |
| UDF คืออะไร? | Python function ที่ register ให้ Spark ใช้กับ DataFrame column ได้ |
| ทำไม UDF มักช้ากว่า built-in functions? | เพราะมักมี serialization overhead และ Spark optimize logic ข้างในได้น้อยลง |
| UDF ต้องระบุอะไรสำคัญ? | Return type เช่น `T.StringType()` และควร handle null |
| ใช้อะไรดู plan เบื้องต้นได้? | `.explain(mode="formatted")` |
| UDF ผิดไหม? | ไม่ผิด แต่ควรใช้เมื่อ built-in / SQL expression ไม่พอจริง ๆ |

## Exercises

### Exercise 1: Build cleaned transaction columns using built-in functions

สร้าง DataFrame ชื่อ `df_ex1_cleaned` จาก `df_transactions`

Requirements:

- สร้าง `customer_name_clean` จาก `customer_name_raw` ด้วย `trim` + `initcap`
- สร้าง `status_clean` จาก `status_raw` ด้วย `trim` + `lower`
- สร้าง `payment_method_clean` โดย normalize space / hyphen เป็น underscore
- สร้าง `is_success` เป็น boolean จาก `status_clean == "success"`
- แสดงผล columns ที่เกี่ยวข้องด้วย `.show(truncate=False)`

Expected result:

- text columns ถูก standardize
- success status ถูก map เป็น boolean ได้

In [12]:
df_ex1_cleaned = (
    df_transactions
    .withColumn("customer_name_clean", F.initcap(F.trim(F.col("customer_name_raw"))))
    .withColumn("status_clean", F.lower(F.trim(F.col("status_raw"))))
    .withColumn(
        "payment_method_clean",
        F.regexp_replace(F.lower(F.trim(F.col("payment_method_raw"))), r"[\s-]+", "_")  # replace one or more spaces/hyphens with underscore
    )
    .withColumn("is_success", F.col("status_clean") == F.lit("success"))
)

df_ex1_cleaned.select(
    "transaction_id",
    "customer_name_raw",
    "customer_name_clean",
    "status_raw",
    "status_clean",
    "payment_method_raw",
    "payment_method_clean",
    "is_success"
).show(truncate=False)

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 29, Finished, Available, Finished, False)

+--------------+-----------------+-------------------+----------+------------+------------------+--------------------+----------+
|transaction_id|customer_name_raw|customer_name_clean|status_raw|status_clean|payment_method_raw|payment_method_clean|is_success|
+--------------+-----------------+-------------------+----------+------------+------------------+--------------------+----------+
|T001          |  Alice Wong     |Alice Wong         | SUCCESS  |success     |Credit Card       |credit_card         |true      |
|T002          |bob smith        |Bob Smith          |success   |success     |prompt-pay        |prompt_pay          |true      |
|T003          |CHARLIE TAN      |Charlie Tan        |Success   |success     |Bank Transfer     |bank_transfer       |true      |
|T004          |Dana Lee         |Dana Lee           |FAILED    |failed      |cash              |cash                |false     |
|T005          |NULL             |NULL               |Pending   |pending     |e wallet    

### Exercise 2: Create risk level using built-in functions and UDF

สร้าง risk level จาก amount ด้วย 2 วิธี:

Requirements:

- built-in column ชื่อ `risk_level_builtin`
  - amount < 0 = `invalid`
  - amount >= 3000 = `high`
  - amount >= 1000 = `medium`
  - otherwise = `low`
- UDF column ชื่อ `risk_level_udf` ด้วย Python function
- compare ว่า result เหมือนกันหรือไม่ด้วย column `is_same_result`

Expected result:

- เห็นว่าทั้ง 2 วิธีให้ผลเหมือนกันได้
- เข้าใจว่า output เหมือนกันไม่ได้แปลว่า UDF เป็นทางเลือกที่ดีที่สุด

In [13]:
def classify_risk_level(amount):
    if amount is None:
        return "unknown"
    if amount < 0:
        return "invalid"
    if amount >= 3000:
        return "high"
    if amount >= 1000:
        return "medium"
    return "low"

classify_risk_level_udf = F.udf(classify_risk_level, T.StringType())

df_ex2_risk = (
    df_transactions
    .withColumn(
        "risk_level_builtin",
        F.when(F.col("amount").isNull(), F.lit("unknown"))
         .when(F.col("amount") < 0, F.lit("invalid"))
         .when(F.col("amount") >= 3000, F.lit("high"))
         .when(F.col("amount") >= 1000, F.lit("medium"))
         .otherwise(F.lit("low"))
    )
    .withColumn("risk_level_udf", classify_risk_level_udf(F.col("amount")))
    .withColumn("is_same_result", F.col("risk_level_builtin") == F.col("risk_level_udf"))
    .select("transaction_id", "amount", "risk_level_builtin", "risk_level_udf", "is_same_result")
)

df_ex2_risk.show()

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 31, Finished, Available, Finished, False)

+--------------+------+------------------+--------------+--------------+
|transaction_id|amount|risk_level_builtin|risk_level_udf|is_same_result|
+--------------+------+------------------+--------------+--------------+
|          T001|1200.5|            medium|        medium|          true|
|          T002| 250.0|               low|           low|          true|
|          T003|3400.0|              high|          high|          true|
|          T004|   0.0|               low|           low|          true|
|          T005| 780.0|               low|           low|          true|
|          T006|5600.0|              high|          high|          true|
|          T007|-100.0|           invalid|       invalid|          true|
|          T008| 980.0|               low|           low|          true|
+--------------+------+------------------+--------------+--------------+



### Exercise 3: Build a summary and write to Lakehouse table

ใช้ `df_payment_grouped` จากตัวอย่างหลักเพื่อสร้าง summary table

Requirements:

- group by `payment_channel_group`
- count transactions
- sum amount
- count success transactions
- write เป็น Lakehouse table ชื่อ `day11_payment_channel_summary`
- read table กลับมาแล้ว `.show()`

Expected result:

- มี table `day11_payment_channel_summary` ใน Lakehouse
- เห็น summary แยกตาม payment channel group

In [14]:
df_ex3_payment_summary = (
    df_payment_grouped
    .groupBy("payment_channel_group")
    .agg(
        F.count("*").alias("transaction_count"),
        F.round(F.sum("amount"), 2).alias("total_amount"),
        F.sum(F.when(F.col("status") == "success", 1).otherwise(0)).alias("success_transaction_count")
    )
    .orderBy("payment_channel_group")
)

(
    df_ex3_payment_summary
    .write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("day11_payment_channel_summary")
)

spark.read.table("day11_payment_channel_summary").show()

StatementMeta(, 6422e428-989d-4bbe-801b-37c6026c9b29, 33, Finished, Available, Finished, False)

+---------------------+-----------------+------------+-------------------------+
|payment_channel_group|transaction_count|total_amount|success_transaction_count|
+---------------------+-----------------+------------+-------------------------+
|                 bank|                2|      4380.0|                        2|
|                 card|                2|      6800.5|                        2|
|      instant_payment|                2|       150.0|                        1|
|                other|                1|         0.0|                        0|
|               wallet|                1|       780.0|                        0|
+---------------------+-----------------+------------+-------------------------+



## Common mistakes

1. **ใช้ UDF ทั้งที่ built-in function ทำได้ง่ายกว่า**

   เช่นใช้ UDF เพื่อ `lower`, `trim`, `replace`, `case when` ทั้งที่ Spark มี function รองรับอยู่แล้ว

2. **ลืมกำหนด return type ให้ UDF**

   UDF ควรระบุ output data type ชัดเจน เช่น `T.StringType()` เพื่อให้ schema คาดเดาได้และอ่านง่าย

3. **ไม่ handle null ใน Python function**

   ถ้า input เป็น null แล้ว function เรียก method เช่น `.strip()` จะ error ทันที

4. **คิดว่า UDF ทำให้ code production-ready ขึ้นเสมอ**

   UDF อาจดูอ่านง่ายใน Python แต่ทำให้ Spark optimize ยากขึ้นและ debug plan ยากขึ้น

5. **ซ่อน business logic สำคัญไว้ใน UDF โดยไม่มี test / comment**

   ถ้า logic ซับซ้อนและจำเป็นต้องใช้ UDF ควรเขียนชื่อ function ให้ชัด, handle edge cases และมี expected output ตรวจได้

6. **ใช้ `.collect()` เพื่อเอาข้อมูลมา loop ด้วย Python แทน Spark functions**

   Pattern นี้ดึงข้อมูลกลับเข้า driver และเสียประโยชน์ของ distributed processing

## Expected Output / Success Criteria

เมื่อจบ Day 11 ควรทำได้:

- อธิบายได้ว่า built-in functions คือ function ที่ Spark engine เข้าใจและ optimize ได้ดีกว่า Python UDF โดยทั่วไป
- ใช้ `F.trim`, `F.lower`, `F.initcap`, `F.regexp_replace`, `F.regexp_extract`, `F.when`, `F.otherwise`, `F.concat` เพื่อ clean / derive columns ได้
- เขียน Python UDF แบบง่าย ๆ ด้วย `F.udf()` และกำหนด return type ได้
- อธิบายได้ว่า UDF ต้อง handle null เอง ไม่ควร assume ว่าทุก input มีค่าเสมอ
- เปรียบเทียบ output ระหว่าง built-in logic และ UDF logic ได้
- ใช้ `.explain(mode="formatted")` เพื่อเริ่มสังเกต plan ของ built-in functions และ Python UDF ได้
- อธิบายได้ว่าทำไม UDF อาจเพิ่ม serialization overhead และทำให้ Spark optimize logic ได้น้อยลง
- เลือกใช้ built-in functions ก่อน UDF เมื่อ logic ยังเขียนด้วย Column expression ได้
- Write output DataFrame เป็น Lakehouse table และ read กลับมาตรวจผลได้ ถ้า attach Lakehouse เรียบร้อย

## Final takeaway

UDF เป็นเครื่องมือที่มีประโยชน์ แต่ไม่ควรเป็น default tool สำหรับทุก transformation:

> ใน Spark pipeline ที่ดี ควรเริ่มจาก built-in functions ก่อน เพราะ Spark เห็น logic ชัดกว่า optimize ได้ดีกว่า และ maintain ง่ายกว่า

สำหรับ Data Engineer สิ่งที่ต้องจำคือ:

- ถ้าเขียนด้วย `pyspark.sql.functions` ได้ ให้ใช้ built-in ก่อน
- ใช้ UDF เมื่อ logic custom จริง ๆ หรือ built-in functions ไม่พอ
- UDF ต้องมี return type และ null handling เสมอ
- อ่าน `.explain()` เพื่อเริ่มเห็นผลกระทบของ UDF ต่อ execution plan
- อย่าใช้ UDF เพื่อแทน `when`, `lower`, `trim`, `regexp_replace` แบบง่าย ๆ โดยไม่จำเป็น

## Optional cleanup

In [ ]:
# spark.sql("DROP TABLE IF EXISTS day11_transaction_function_features")
# spark.sql("DROP TABLE IF EXISTS day11_payment_channel_summary")